<a href="https://colab.research.google.com/github/Rams302/TitanicAPI/blob/main/TitanicAutoML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# TITANIC DATASET - AUTOGLUON AUTOML + MLOPS
# =====================================================


# =====================================================
# INSTALACIÓN DE DEPENDENCIAS
# =====================================================

!pip -q install autogluon
!pip -q install mlflow



# =====================================================
# IMPORTS
# =====================================================


import pandas as pd
import json
import shutil

from datetime import datetime

from google.colab import files


from autogluon.tabular import TabularPredictor


import mlflow



# =====================================================
# CONFIGURACIÓN DEL PROYECTO
# =====================================================


PROJECT_NAME = "TitanicAPI"

MODEL_NAME = "TitanicAutoML"


TRAINING_DATE = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)



# =====================================================
# CARGAR DATASET
# =====================================================


df = pd.read_csv(
    "train.csv"
)



print(df.head())


print(df.info())



# =====================================================
# LIMPIEZA DE DATOS
# =====================================================


df = df.drop(

    columns=[

        "PassengerId",

        "Name",

        "Ticket",

        "Cabin"

    ]

)



print(
    "Dataset preparado correctamente"
)



# =====================================================
# SEPARACIÓN TRAIN / TEST
# =====================================================


train = df.sample(

    frac=0.8,

    random_state=42

)



test = df.drop(

    train.index

)



print()

print(
    "Train:",
    train.shape
)


print(
    "Test:",
    test.shape
)



# =====================================================
# ENTRENAMIENTO AUTOGLUON
# =====================================================


predictor = TabularPredictor(

    label="Survived",

    eval_metric="accuracy"

).fit(

    train_data=train,

    presets="medium_quality",

    hyperparameters={

        "GBM": {},

        "RF": {},

        "XT": {},

        "CAT": {},

        "XGB": {}

    }

)



print(
    "Entrenamiento terminado"
)



# =====================================================
# LEADERBOARD
# =====================================================


leaderboard = predictor.leaderboard(

    test

)


leaderboard



# =====================================================
# MEJOR MODELO
# =====================================================


best_model = predictor.model_best



print()

print(
    "Mejor modelo:"
)


print(
    best_model
)



# =====================================================
# EVALUACIÓN
# =====================================================


performance = predictor.evaluate(

    test

)



print()

print(
    performance
)



accuracy = performance["accuracy"]



# =====================================================
# GENERAR MÉTRICAS PARA MLOPS
# =====================================================


model_metrics = {


    "model_name":
        MODEL_NAME,


    "framework":
        "AutoGluon",


    "framework_version":
        "1.4.0",


    "best_model":
        best_model,


    "evaluation_metric":
        "accuracy",


    "accuracy":
        float(accuracy),


    "training_rows":
        int(len(train)),


    "testing_rows":
        int(len(test)),


    "training_date":
        TRAINING_DATE


}



with open(

    "model_metrics.json",

    "w"

) as file:


    json.dump(

        model_metrics,

        file,

        indent=4

    )



print(
    "Archivo model_metrics.json generado"
)



# =====================================================
# VISUALIZAR MÉTRICAS
# =====================================================


with open(
    "model_metrics.json"
) as file:


    metrics = json.load(file)



metrics



# =====================================================
# REGISTRO MLflow
# =====================================================


mlflow.set_experiment(

    PROJECT_NAME

)



with mlflow.start_run():


    mlflow.log_param(

        "framework",

        "AutoGluon"

    )


    mlflow.log_param(

        "best_model",

        best_model

    )


    mlflow.log_param(

        "preset",

        "medium_quality"

    )


    mlflow.log_metric(

        "accuracy",

        float(accuracy)

    )


    mlflow.log_metric(

        "training_rows",

        len(train)

    )


    mlflow.log_metric(

        "testing_rows",

        len(test)

    )


    mlflow.log_artifact(

        "model_metrics.json"

    )



print(
    "Experimento registrado en MLflow"
)



# =====================================================
# GUARDAR MODELO
# =====================================================


predictor.save()



print(
    "Modelo guardado correctamente"
)



# =====================================================
# COMPRIMIR MODELO
# =====================================================


shutil.make_archive(

    "TitanicModel",

    "zip",

    predictor.path

)



print(
    "TitanicModel.zip creado"
)



# =====================================================
# DESCARGAR MODELO
# =====================================================


files.download(

    "TitanicModel.zip"

)